<a href="https://colab.research.google.com/github/AyushKhatri-Dev/flyrank-search-intelligence/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AyushKhatri-Dev/flyrank-search-intelligence/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*


Lane 4 is a **ranking / scoring** problem, built on top of a **regression** step.

The regression step predicts what CTR a page would normally get, given what was
true about it before anyone clicked: its average position, how many impressions
it received, its content type, its intent, its length, its age. The ranking step
then orders pages by how far their actual CTR falls below that prediction.

It is not classification, because I am not labelling pages as "good" or "bad" —
I am ordering them. It is not clustering, because I already know what I am
looking for.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**The regression target is observed CTR** — clicks divided by impressions over
the 90-day window. This is measured in the data, not defined by a rule.

**The ranked output is the residual**: actual CTR minus predicted CTR. A large
negative residual means the page under-captured clicks relative to pages that
look like it.

**Where the proxy weakness is.** The residual is a scoring quantity I construct,
not an outcome I observed. It tells me a page looks unusual today; it does not
tell me the page later improved. The starter dataset is a single aggregated
90-day snapshot, so it has no future window to check against.

The stronger target, once I move to the warehouse daily table, is a future
observed outcome: features from a prior window, then whether CTR actually rose
in a later window. I am treating the residual as a **provisional proxy** and
will say so in every claim built on it.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Model level, computable today:** mean absolute error on predicted CTR, against
a baseline that just predicts each position tier's median CTR. If the model
cannot beat that simple rule, the extra complexity has not earned its place.

**Decision level, the metric that actually matters:** precision@50 — of the 50
pages the ranking puts on top, how many turn out to be real opportunities. This
matches how the list is used: a reviewer works down from the top, not across all
30,000 pages.

Precision@50 needs a ground truth I do not have on the starter data, because
there is no later window to check. Until I move to the warehouse, I will judge
the ranking three ways instead: how much it differs from the naive lowest-CTR
list, whether it stays balanced across position tiers, and a hand review of the
top 20 pages.

I am naming these metrics now, before training anything.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content page.** The lane slice has 12,009 rows and 12,009
unique `content_id` values, so the grain is confirmed — there is no hidden
duplication to collapse before modelling.

The slice is every page with at least 500 impressions in the 90-day window and
an average position between 1 and 20. Below that bar a CTR gap is mostly noise,
and past position 20 there is little to act on.

**Three things I checked before trusting this slice:**

**CTR is in percentage points, not a fraction.** Row 0 has 29 clicks on 3,803
impressions, which is 0.76% — and the `ctr` column reads 0.76. So the tier
medians I found earlier (0.23, 0.24, 0.17) are percentages, not proportions.
Worth adding: even the top tier sits well below 1%, far under published CTR
curves for position 1-3. This is an anonymized slice, so I will compare pages
only against each other and never against public benchmarks.

**`word_count` is missing for 3,658 of 12,009 pages — about 30%.** That is too
much to drop and too much to fill with a single value without distorting the
feature. I will either treat missingness as its own signal or leave the column
out of the first model and test whether adding it helps. `main_intent` is
missing on 163 rows (1.4%), which is small enough to handle simply.

**The client distribution is very uneven.** 28 clients hold the 12,009 pages,
but the median client has 95 pages while the largest has 4,391 — roughly a
third of the entire slice. A plain random train/test split would let that one
client dominate both sides, and the model could score well by learning that
client's habits rather than a general pattern. Client-holdout validation is
therefore not optional in this lane; it is the only honest option.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# --- The lane slice: visible pages only ---
lane = df[(df["impressions_90d"] >= 500) & (df["avg_position"].between(1, 20))].copy()

# --- Unit of analysis: one row = one content page ---
print(f"Rows in lane slice: {len(lane):,}")
print(f"Unique content_id:  {lane['content_id'].nunique():,}")
print(f"Clients covered:    {lane['client_id'].nunique():,}")

# --- Which columns are actually available for the expected-CTR model ---
wanted = ["content_id", "client_id", "content_type", "main_intent",
          "impressions_90d", "clicks_90d", "avg_position", "ctr",
          "word_count", "content_age_days", "days_since_last_update"]
available = [c for c in wanted if c in lane.columns]
print("\nMissing from starter data:", [c for c in wanted if c not in lane.columns])

# --- Sketch of the target column ---
lane["position_tier"] = pd.cut(lane["avg_position"], [0, 3, 10, 20],
                               labels=["1-3", "4-10", "11-20"])
lane["expected_ctr_rule"] = lane.groupby("position_tier", observed=True)["ctr"].transform("median")
lane["ctr_residual"] = lane["ctr"] - lane["expected_ctr_rule"]

print("\nOne row = one content page. First 5 rows of the lane slice:")
lane[available + ["position_tier", "expected_ctr_rule", "ctr_residual"]].head()

print("CTR check — row 0:", lane["clicks_90d"].iloc[0], "/", lane["impressions_90d"].iloc[0],
      "=", round(lane["clicks_90d"].iloc[0] / lane["impressions_90d"].iloc[0] * 100, 2),
      "vs ctr column:", lane["ctr"].iloc[0])

print("\nMissing values in planned features:")
print(lane[["word_count", "content_age_days", "days_since_last_update",
            "content_type", "main_intent", "avg_position"]].isna().sum())

print("\nPages per client — smallest and largest:")
print(lane["client_id"].value_counts().describe())


Rows in lane slice: 12,009
Unique content_id:  12,009
Clients covered:    28

Missing from starter data: []

One row = one content page. First 5 rows of the lane slice:
CTR check — row 0: 29 / 3803 = 0.76 vs ctr column: 0.76

Missing values in planned features:
word_count                3658
content_age_days             0
days_since_last_update       0
content_type                 0
main_intent                163
avg_position                 0
dtype: int64

Pages per client — smallest and largest:
count      28.000000
mean      428.892857
std       886.938611
min         3.000000
25%        17.000000
50%        95.500000
75%       471.500000
max      4391.000000
Name: count, dtype: float64


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*


My first attempt at this was a fixed rule: take each position tier's median CTR
and rank pages by how far below it they sit. It produced a list, and the list
was clearly biased — all 50 top-ranked pages came from tier 4-10. The rule uses
an absolute difference from a tier median, and tier 4-10 has the highest median,
so it can generate the largest negative gaps by construction.

That is not a bug I can patch with a better threshold. It is what happens when
one signal decides everything.

What a page's CTR should be depends on several things at once: where it ranks,
how many impressions it gets, whether the intent is informational or
transactional, what content type it is, how old it is. These interact. A
transactional page at position 5 and an informational page at position 5 do not
have the same normal. Writing that by hand means writing a separate rule for
every combination, and each threshold would be a guess.

A regression model can estimate expected CTR from all of those signals together
and give me a residual that is comparable across the whole slice. That is the
specific job I need done, and it is why ML earns its place here.

**But this is a claim I have to test, not assume.** My baseline is the tier-
median rule I already built. If the model cannot beat it on prediction error,
or if its ranking is not meaningfully different, then the rule was good enough
and I should say so. Naming that comparison now is the point — a model that is
only judged after the fact always looks like it won.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.